# 07 Leaderboard

In [ ]:
import zipfile

import pandas as pd
from scipy.stats import spearmanr
from kaggle.api.kaggle_api_extended import KaggleApi

In [2]:
api = KaggleApi()
api.authenticate()
subs = api.competition_submissions("ieee-fraud-detection", page_size=100)

log = pd.DataFrame([{
    "file": s.file_name,
    "date": s.date,
    "public": pd.to_numeric(s.public_score),
    "private": pd.to_numeric(s.private_score),
} for s in subs]).sort_values("date")
log

,file,date,public,private
3,06_ensemble.csv,2026-09-27 10:52:59.903,0.952221,0.926361
2,04_lgb_engineered.csv,2026-09-27 10:53:04.483,0.950279,0.921902
1,05_nn.csv,2026-09-27 10:53:08.887,0.932374,0.902242
0,04_lgb_raw.csv,2026-09-27 10:53:14.040,0.932385,0.904201


In [3]:
lb = log.drop_duplicates("file", keep="last").set_index("file")[["public", "private"]]

## Final models

In [4]:
gbdt = pd.read_csv("../data/interim/04_models.csv", index_col="model")
nn = pd.read_csv("../data/interim/05_runs.csv", index_col=0)
ens = pd.read_csv("../data/interim/06_combiners.csv", index_col=0)

final = pd.DataFrame({
    "CV AUC": {
        "LightGBM, raw features": gbdt.loc["lgb_raw", "CV AUC"],
        "neural net": nn.loc[nn["selected"], "CV AUC"].iloc[0],
        "LightGBM, engineered": gbdt.loc["lgb_engineered", "CV AUC"],
        "ensemble": ens["CV AUC"].max(),
    },
    "file": {
        "LightGBM, raw features": "04_lgb_raw.csv",
        "neural net": "05_nn.csv",
        "LightGBM, engineered": "04_lgb_engineered.csv",
        "ensemble": "06_ensemble.csv",
    },
})
final = final.join(lb, on="file").drop(columns="file")
final["CV - private"] = final["CV AUC"] - final["private"]
final["public - private"] = final["public"] - final["private"]
print("Spearman CV vs private:", round(spearmanr(final["CV AUC"], final["private"]).correlation, 2))
final.round(4)

Spearman CV vs private: 1.0


,CV AUC,public,private,CV - private,public - private
"LightGBM, raw features",0.9109,0.9324,0.9042,0.0067,0.0282
neural net,0.9068,0.9324,0.9022,0.0046,0.0301
"LightGBM, engineered",0.9345,0.9503,0.9219,0.0126,0.0284
ensemble,0.9367,0.9522,0.9264,0.0104,0.0259


CV ranks the four models exactly like private and is optimistic by about 0.01.

## Place on the leaderboard

In [5]:
api.competition_leaderboard_download("ieee-fraud-detection", "../data/interim")
with zipfile.ZipFile("../data/interim/ieee-fraud-detection.zip") as z:
    public_lb = pd.read_csv(z.open(z.namelist()[0]))

final["public place"] = [(public_lb["Score"] > s).sum() + 1 for s in final["public"]]
print("teams:", len(public_lb))
final[["public", "public place"]]

teams: 6355


,public,public place
"LightGBM, raw features",0.932385,4276
neural net,0.932374,4277
"LightGBM, engineered",0.950279,1874
ensemble,0.952221,1510


The ensemble's 0.9264 on private would be about 1276th of 6355, the top 20%.

## Summary

- The final ensemble gets 0.9264 on private and 0.9522 on public.
- My features and the ensemble both help on the real test.
- Time-block CV ranked the models exactly like private.